In [ ]:
import torch
from torch import nn
import d2l.torch as d2l

# ===================== 1. 生成人造数据集（高阶多项式，天然容易过拟合） =====================
n_train, n_test = 20, 100  # 训练样本很少，制造过拟合场景
true_w = torch.tensor([0.05, 1.0, -2.2, 1.1])  # 4阶多项式真实权重
true_b = 0.08

# 构造特征：x, x^2, x^3, x^4
def poly_features(x):
    features = torch.ones((x.shape[0], len(true_w)))
    for i in range(1, len(true_w)):
        features[:, i] = x ** i
    return features

# 生成带噪声数据
torch.manual_seed(42)
x_train = torch.randn(n_train)
x_test = torch.randn(n_test)
train_features = poly_features(x_train)
test_features = poly_features(x_test)

train_labels = train_features @ true_w + true_b + torch.normal(0, 0.1, (n_train,))
test_labels = test_features @ true_w + true_b + torch.normal(0, 0.1, (n_test,))

# ===================== 2. 定义评估损失函数 =====================
def evaluate_loss(net, data_iter, loss):
    metric = d2l.Accumulator(2)
    for X, y in data_iter:
        out = net(X)
        y = y.reshape(out.shape)
        l = loss(out, y)
        metric.add(l.sum(), l.numel())
    return metric[0] / metric[1]

# ===================== 3. 训练封装函数 =====================
def train(train_features, test_features, train_labels, test_labels,
          lr=0.01, num_epochs=100, weight_decay=0):
    batch_size = 5
    train_iter = d2l.load_array((train_features, train_labels.reshape(-1,1)), batch_size)
    test_iter = d2l.load_array((test_features, test_labels.reshape(-1,1)), batch_size, is_train=False)

    # 线性回归模型
    net = nn.Sequential(nn.Linear(train_features.shape[1], 1))
    # 初始化权重
    for layer in net:
        nn.init.normal_(layer.weight, mean=0, std=0.01)
        nn.init.constant_(layer.bias, 0)

    loss = nn.MSELoss()
    # 关键：weight_decay 就是L2权重衰退！
    trainer = torch.optim.SGD(net.parameters(), lr=lr, weight_decay=weight_decay)

    animator = d2l.Animator(xlabel='epoch', ylabel='loss', yscale='log',
                            xlim=[1, num_epochs], legend=['train loss', 'test loss'])
    
    for epoch in range(num_epochs):
        for X, y in train_iter:
            trainer.zero_grad()
            l = loss(net(X), y)
            l.backward()
            trainer.step()
        if (epoch+1) % 5 == 0:
            animator.add(epoch+1, (evaluate_loss(net, train_iter, loss),
                                   evaluate_loss(net, test_iter, loss)))
    print('最终权重：', net[0].weight.data)

# ===================== 4. 两种对比实验 =====================
# ① 不使用权重衰退（weight_decay=0），观察过拟合
train(train_features, test_features, train_labels, test_labels,
      lr=0.01, num_epochs=150, weight_decay=0)